# Lime Stability Example

We use the boston dataset to build a regression model (Random Forest),
afterwards we employ Lime on a specific unit of the dataset 
and we test its stability using the CSI and VSI indices.

In [ ]:
from sklearn.datasets import load_boston
import sklearn.ensemble
import sklearn.model_selection
import numpy as np

In [ ]:
from lime_stability.stability import LimeTabularExplainerOvr

In [ ]:
# Set the seed
seed = 100
np.random.RandomState(seed)
np.random.seed(seed)

In [ ]:
boston = load_boston()

In [ ]:
boston.data.shape

In [ ]:
rf = sklearn.ensemble.RandomForestRegressor(n_estimators=1000)

In [ ]:
train, test, labels_train, labels_test = sklearn.model_selection.train_test_split(boston.data, boston.target, train_size=0.80, test_size=0.20)


In [ ]:
rf.fit(train, labels_train)

In [ ]:
print('
      Random Forest MSError', np.mean((rf.predict(test) - labels_test) ** 2))

In [ ]:
print('MSError when predicting the mean', np.mean((labels_train.mean() - labels_test) ** 2))

In [ ]:
categorical_features = np.argwhere(np.array([len(set(boston.data[:,x])) for x in range(boston.data.shape[1])]) <= 10).flatten()
categorical_features

In [ ]:
explainer = LimeTabularExplainerOvr(train, feature_names=boston.feature_names, class_names=['price'], categorical_features=categorical_features, verbose=True, mode='regression')

Set the parameters to be used in the explain_instance method 

In [ ]:
i = 25

display(test[i])

params = {"data_row": test[i],
          "predict_fn": rf.predict,
          "num_samples": 5000,
          "num_features": 10,
          "distance_metric": "euclidean"}

In [ ]:
exp = explainer.explain_instance(**params)

In [ ]:
exp.show_in_notebook(show_table=True)

In [ ]:
exp.as_list()

## Evaluating LIME Stability

We use the method check_stability to evaluate the stability of the explanation given before for the data point 25

In [ ]:
csi, vsi = explainer.check_stability(n_calls=10,**params,index_verbose=False)
print("CSI: ",csi,"\nVSI: ",vsi,"\n")    